# 🛰️ Detección de Barcos en Imágenes Satelitales con YOLO11

## Trabajo Práctico — Diplomatura en Inteligencia Artificial: Computer Vision

---

### Objetivo
Entrenar un modelo de detección de objetos (YOLO11) para identificar **barcos** en **imágenes satelitales**, utilizando el dataset de cuerpos de agua de Kaggle como fuente de imágenes base y Roboflow como plataforma de anotación y gestión del dataset.

### Pipeline completo
```
Kaggle Dataset          Roboflow               YOLO11 Training        Deploy
(Imágenes satélite) --> (Anotación de barcos) --> (Fine-tuning) ------> (API)
```

### ¿Por qué YOLO11?
YOLO11 (Ultralytics, 2024) es la arquitectura más reciente de la familia YOLO. Respecto a versiones anteriores:
- **22% menos parámetros** que YOLOv8m con igual o mayor mAP en COCO
- Mejor extracción de features en objetos pequeños (crítico para barcos desde satélite)
- Disponible en 5 tamaños: n, s, m, l, x
- Soporte nativo de exportación a TensorRT, ONNX, CoreML

### ¿Por qué es desafiante detectar barcos en imágenes satelitales?
1. **Escala pequeña**: un barco puede ocupar apenas 20×10 píxeles en una imagen de 640×640
2. **Variabilidad de orientación**: los barcos se capturan desde arriba, en cualquier ángulo
3. **Contexto visual complejo**: reflejos de agua, nubes, costas
4. **Densidad variable**: puertos con decenas de barcos vs. mar abierto con uno solo

> **Nota sobre el dataset:** El dataset de Kaggle contiene imágenes satelitales clasificadas en categorías de cuerpos de agua (mar, lago, río, pantano), pero **no incluye anotaciones de barcos**. Por eso el flujo incluye un paso de anotación manual/semi-automática en Roboflow antes de entrenar.

---
## Sección 0: Verificación del entorno

Antes de comenzar, verificar que Colab asignó una GPU. Si no aparece GPU, ir a:
**Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)**

In [ ]:
# Verificar GPU disponible
!nvidia-smi

import os
HOME = os.getcwd()
print(f"\n✅ Directorio de trabajo: {HOME}")

---
## Sección 1: Instalación de dependencias

Instalamos:
- **ultralytics**: librería oficial de YOLO11
- **supervision**: utilidades de visualización y anotación
- **roboflow**: cliente Python para gestión de datasets y deploy
- **kaggle**: cliente para descargar datasets de Kaggle

In [ ]:
%pip install ultralytics supervision roboflow kaggle --quiet
!yolo settings sync=False  # Deshabilitar telemetría

import ultralytics
ultralytics.checks()  # Verificar instalación y GPU

---
## Sección 2: Configuración de credenciales

### 2.1 Roboflow API Key
1. Ir a [roboflow.com](https://roboflow.com) → Settings → API Keys
2. En Colab: **Secrets** (ícono 🔑 en el panel izquierdo) → Agregar `ROBOFLOW_API_KEY`

### 2.2 Kaggle API Key  
1. Ir a [kaggle.com](https://kaggle.com) → Account → API → **Create New Token**
2. Descarga `kaggle.json`
3. Subir el archivo a Colab usando el código a continuación

In [ ]:
from google.colab import userdata, files
import os, shutil

# --- Roboflow ---
ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
print("✅ Roboflow API Key cargada")

# --- Kaggle ---
# Opción A: subir kaggle.json desde tu computadora
print("\n📁 Sube el archivo kaggle.json:")
uploaded = files.upload()  # Se abrirá un diálogo de subida de archivo

os.makedirs('/root/.kaggle', exist_ok=True)
shutil.copy('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print("✅ Kaggle API Key configurada")

---
## Sección 3: Descarga y exploración del dataset de Kaggle

El dataset **"Satellite Images of Water Bodies"** (franciscoescobar) contiene ~2,800 imágenes satelitales de resolución media, organizadas en 4 categorías:

| Categoría | Descripción | Relevancia para barcos |
|-----------|-------------|------------------------|
| `sea`     | Océano y mar abierto | ⭐⭐⭐ Alta — mayor probabilidad de barcos |
| `lake`    | Lagos       | ⭐⭐ Media — barcos pequeños |
| `river`   | Ríos        | ⭐ Baja — posibles barcazas |
| `swamp`   | Pantanos    | ✗ Sin barcos típicamente |

**Estrategia**: usaremos principalmente las imágenes de `sea` para anotar y entrenar, ya que son las que mayor variedad de embarcaciones presentan.

In [ ]:
import kaggle
import os

DATASETS_DIR = f'{HOME}/datasets'
KAGGLE_DIR = f'{DATASETS_DIR}/water_bodies'
os.makedirs(KAGGLE_DIR, exist_ok=True)

print("⬇️  Descargando dataset de Kaggle...")
!kaggle datasets download -d franciscoescobar/satellite-images-of-water-bodies \
    -p {KAGGLE_DIR} --unzip

print("\n📂 Estructura del dataset:")
!find {KAGGLE_DIR} -type d | head -20

print("\n📊 Cantidad de imágenes por categoría:")
for category in ['sea', 'lake', 'river', 'swamp']:
    path = f'{KAGGLE_DIR}/water_body_dataset/water_bodies/{category}'
    if os.path.exists(path):
        count = len([f for f in os.listdir(path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        print(f"  {category:10s}: {count} imágenes")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob, random

# Visualizar ejemplos de imágenes del mar (donde buscaremos barcos)
sea_images = glob.glob(f'{KAGGLE_DIR}/**/sea/*.jpg', recursive=True) + \
             glob.glob(f'{KAGGLE_DIR}/**/sea/*.png', recursive=True)

print(f"🌊 Total de imágenes de mar encontradas: {len(sea_images)}")

# Mostrar 6 imágenes de ejemplo
sample = random.sample(sea_images, min(6, len(sea_images)))

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Ejemplos de imágenes satelitales de mar — Dataset Kaggle', fontsize=14)

for ax, img_path in zip(axes.flat, sample):
    img = mpimg.imread(img_path)
    ax.imshow(img)
    ax.set_title(os.path.basename(img_path), fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.show()

# Estadísticas de dimensiones
from PIL import Image
import numpy as np

print("\n📐 Estadísticas de dimensiones (muestra de 50 imágenes):")
sample_check = random.sample(sea_images, min(50, len(sea_images)))
widths, heights = [], []
for p in sample_check:
    try:
        w, h = Image.open(p).size
        widths.append(w)
        heights.append(h)
    except:
        pass

print(f"  Ancho  — min: {min(widths)}, max: {max(widths)}, promedio: {np.mean(widths):.0f}px")
print(f"  Alto   — min: {min(heights)}, max: {max(heights)}, promedio: {np.mean(heights):.0f}px")

---
## Sección 4: Anotación de imágenes con Roboflow

### El problema fundamental
El dataset de Kaggle tiene imágenes satelitales pero **sin anotaciones de barcos**. Para entrenar un detector de objetos necesitamos *bounding boxes* alrededor de cada barco.

### Flujo de anotación en Roboflow (hacerlo manualmente en la web)

**Paso 1:** Crear un proyecto en [app.roboflow.com](https://app.roboflow.com)
- New Project → Object Detection
- Nombre: `ship-detection-satellite`
- Clase: `ship` (solo una clase por ahora)

**Paso 2:** Subir imágenes del mar
- Upload → seleccionar las imágenes de la carpeta `sea/` del dataset de Kaggle
- Recomendado: **200-500 imágenes** para un primer modelo funcional

**Paso 3:** Anotar barcos
- Usar la herramienta de bounding box de Roboflow
- **Tip de anotación**: dibujar el rectángulo ajustado al casco del barco, excluyendo estelas de agua
- **Roboflow Auto-Label**: usar el botón *Auto Label* con un modelo fundacional (grounding DINO o SAM) para acelerar el proceso

**Paso 4:** Generar dataset con aumentación
- Train/Valid/Test split: 70% / 20% / 10%
- Augmentations: activar Rotate, Flip, Brightness, Blur
- Generate → Export → **YOLOv11** format → Get API snippet

> ⚡ **Alternativa rápida**: usar un dataset de detección de barcos ya anotado de Roboflow Universe, como:
> - [SAR Ship Detection](https://universe.roboflow.com/roboflow-100/aerial-ships-r24io)
> - [Ships Dataset](https://universe.roboflow.com/ships-yolo)
> Luego mezclarlo con las imágenes del dataset de Kaggle anotadas.

In [ ]:
# OPCIONAL: Subir imágenes automáticamente a Roboflow via API
# Ejecutar esto si preferís subir las imágenes programáticamente
# en lugar de hacerlo manualmente desde la web de Roboflow

from roboflow import Roboflow
import glob, random

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

# --- CONFIGURAR CON TUS DATOS ---
WORKSPACE_NAME = "tu-workspace"       # ← reemplazar con tu workspace en Roboflow
PROJECT_NAME = "ship-detection-satellite"  # ← nombre del proyecto que creaste
# --------------------------------

project = rf.workspace(WORKSPACE_NAME).project(PROJECT_NAME)

# Subir imágenes de mar (muestra de 300 imágenes)
sea_images = glob.glob(f'{KAGGLE_DIR}/**/sea/*.jpg', recursive=True)
sample_to_upload = random.sample(sea_images, min(300, len(sea_images)))

print(f"📤 Subiendo {len(sample_to_upload)} imágenes a Roboflow...")
for i, img_path in enumerate(sample_to_upload):
    project.upload(img_path, batch_name='kaggle-sea-batch-1')
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{len(sample_to_upload)} subidas")

print("✅ Subida completada. Ahora anotá los barcos en app.roboflow.com")

---
## Sección 5: Descarga del dataset anotado desde Roboflow

Una vez que anotaste las imágenes y generaste el dataset en Roboflow, descargarlo desde aquí.
Roboflow genera automáticamente el dataset en el formato correcto para YOLO11 (carpetas `train/`, `valid/`, `test/` + `data.yaml`).

In [ ]:
from roboflow import Roboflow
from google.colab import userdata
import os

ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
rf = Roboflow(api_key=ROBOFLOW_API_KEY)

%cd {HOME}/datasets

# --- CONFIGURAR CON TUS DATOS DE ROBOFLOW ---
WORKSPACE_NAME = "tu-workspace"            # ← tu workspace
PROJECT_NAME   = "ship-detection-satellite" # ← nombre del proyecto
VERSION_NUMBER = 1                          # ← número de versión generada
# --------------------------------------------

workspace = rf.workspace(WORKSPACE_NAME)
project   = workspace.project(PROJECT_NAME)
version   = project.version(VERSION_NUMBER)

dataset = version.download("yolov11")

print(f"\n✅ Dataset descargado en: {dataset.location}")
print(f"   Versión: {dataset.version}")

# Verificar estructura
!ls {dataset.location}
print("\n📋 Contenido de data.yaml:")
!cat {dataset.location}/data.yaml

In [ ]:
# Estadísticas del dataset anotado
import os, glob

splits = ['train', 'valid', 'test']
total_annotations = 0

print("📊 Estadísticas del dataset anotado:")
print("-" * 40)

for split in splits:
    img_dir   = f"{dataset.location}/{split}/images"
    label_dir = f"{dataset.location}/{split}/labels"

    if not os.path.exists(img_dir):
        continue

    images = glob.glob(f"{img_dir}/*.jpg") + glob.glob(f"{img_dir}/*.png")
    labels = glob.glob(f"{label_dir}/*.txt")

    # Contar anotaciones totales
    ann_count = 0
    for lf in labels:
        with open(lf) as f:
            lines = [l.strip() for l in f if l.strip()]
            ann_count += len(lines)

    total_annotations += ann_count
    print(f"  {split:8s}: {len(images):4d} imágenes, {ann_count:5d} barcos anotados")

print("-" * 40)
print(f"  TOTAL   :         {total_annotations} anotaciones")

# Calcular promedio de objetos por imagen
train_labels = glob.glob(f"{dataset.location}/train/labels/*.txt")
if train_labels:
    counts = []
    for lf in train_labels:
        with open(lf) as f:
            counts.append(len([l for l in f if l.strip()]))
    import numpy as np
    print(f"\n  Barcos/imagen (train) — promedio: {np.mean(counts):.2f}, max: {max(counts)}")

---
## Sección 6: Entrenamiento de YOLO11 para detección de barcos

### Decisiones de diseño del entrenamiento

| Parámetro | Notebook original | Esta versión | Justificación |
|-----------|------------------|--------------|---------------|
| `model`   | yolo11s (small)  | **yolo11m (medium)** | Barcos satelitales son objetos pequeños; más capacidad del modelo → mejor precisión |
| `epochs`  | 10               | **100** | 10 épocas es insuficiente; convergencia real requiere 80-150 épocas |
| `imgsz`   | 640              | **640** | Balance entre detección de objetos pequeños y memoria GPU (T4 tiene 15GB) |
| `patience` | —               | **25** | Early stopping para evitar overfitting |
| `batch`   | auto             | **16** | Tamaño estable para T4; más estabilidad en gradientes |
| `lr0`     | 0.01             | **0.01** | Learning rate inicial estándar para fine-tuning |
| `augment` | default          | **custom** | Rotación ±30°, flips H+V, escala — clave para imágenes satelitales |
| `cos_lr`  | —               | **True** | Cosine learning rate scheduler para mejor convergencia |

### ¿Por qué `yolo11m` y no `yolo11n`?
Los modelos **nano (n)** y **small (s)** tienen menor capacidad de feature extraction.
Para objetos pequeños en imágenes satelitales (barcos de 10-30px en una imagen de 640px),
el modelo **medium (m)** ofrece un mejor balance entre velocidad y precisión.
El modelo **large (l)** sería ideal pero requiere más VRAM (puede dar OOM en T4).

### ¿Por qué importa la aumentación de datos en imágenes satelitales?
- Los satélites capturan desde distintos ángulos → **rotación aleatoria**
- Un barco navegando norte vs. sur es igual de válido → **flips horizontales y verticales**
- Barcos en diferentes zooms del satélite → **escala aleatoria**
- Condiciones de luz variable → **ajuste de brillo/contraste**

In [ ]:
import os
%cd {HOME}

# Entrenamiento YOLO11 optimizado para detección de barcos en imágenes satelitales
!yolo task=detect mode=train \
    model=yolo11m.pt \
    data={dataset.location}/data.yaml \
    epochs=100 \
    imgsz=640 \
    batch=16 \
    patience=25 \
    cos_lr=True \
    lr0=0.01 \
    lrf=0.01 \
    momentum=0.937 \
    weight_decay=0.0005 \
    warmup_epochs=3 \
    degrees=30.0 \
    flipud=0.5 \
    fliplr=0.5 \
    scale=0.5 \
    shear=2.0 \
    mosaic=1.0 \
    mixup=0.1 \
    copy_paste=0.1 \
    plots=True \
    name=ship_detection_v1 \
    project={HOME}/runs/detect

print("\n✅ Entrenamiento completado!")
print(f"📁 Resultados guardados en: {HOME}/runs/detect/ship_detection_v1/")

### Explicación de los hiperparámetros de aumentación

| Parámetro | Valor | Qué hace |
|-----------|-------|----------|
| `degrees=30.0` | ±30° | Rota imágenes aleatoriamente. **Clave para satélite**: un barco orientado en cualquier dirección |
| `flipud=0.5` | 50% prob | Voltea verticalmente. En satélite no hay "arriba" absoluto |
| `fliplr=0.5` | 50% prob | Voltea horizontalmente. Misma razón |
| `scale=0.5` | ±50% zoom | Simula diferentes altitudes del satélite |
| `shear=2.0` | ±2° | Deformación leve para simular perspectiva |
| `mosaic=1.0` | 100% | Combina 4 imágenes en una — mejora detección de objetos pequeños significativamente |
| `mixup=0.1` | 10% prob | Mezcla imágenes superpuestas — regularización adicional |
| `copy_paste=0.1` | 10% prob | Copia objetos entre imágenes — aumenta diversidad de instancias de barcos |

---
## Sección 7: Análisis de resultados del entrenamiento

In [ ]:
TRAIN_DIR = f'{HOME}/runs/detect/ship_detection_v1'
!ls {TRAIN_DIR}

In [ ]:
from IPython.display import Image as IPyImage

print("📊 Matriz de confusión normalizada:")
IPyImage(filename=f'{TRAIN_DIR}/confusion_matrix_normalized.png', width=600)

In [ ]:
print("📈 Curvas de entrenamiento (loss, métricas):")
IPyImage(filename=f'{TRAIN_DIR}/results.png', width=900)

In [ ]:
print("🔍 Predicciones en batch de validación:")
IPyImage(filename=f'{TRAIN_DIR}/val_batch0_pred.jpg', width=900)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Leer CSV de métricas de entrenamiento
results_csv = f'{TRAIN_DIR}/results.csv'

if os.path.exists(results_csv):
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()

    # Mostrar métricas finales
    last = df.iloc[-1]
    best_epoch = df['metrics/mAP50(B)'].idxmax()
    best = df.iloc[best_epoch]

    print("📈 Métricas finales del entrenamiento:")
    print(f"  Época final          : {int(last.get('epoch', len(df)))}")
    print(f"  mAP@50 (mejor época) : {best.get('metrics/mAP50(B)', 0):.4f} (época {best_epoch+1})")
    print(f"  mAP@50-95 (mejor)    : {best.get('metrics/mAP50-95(B)', 0):.4f}")
    print(f"  Precisión (mejor)    : {best.get('metrics/precision(B)', 0):.4f}")
    print(f"  Recall (mejor)       : {best.get('metrics/recall(B)', 0):.4f}")

    # Graficar curvas de mAP
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(df.index+1, df.get('metrics/mAP50(B)', [0]*len(df)), label='mAP@50', color='blue')
    axes[0].plot(df.index+1, df.get('metrics/mAP50-95(B)', [0]*len(df)), label='mAP@50-95', color='orange')
    axes[0].set_title('Métricas de Detección por Época')
    axes[0].set_xlabel('Época')
    axes[0].set_ylabel('mAP')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(df.index+1, df.get('train/box_loss', [0]*len(df)), label='train/box_loss', color='red')
    axes[1].plot(df.index+1, df.get('val/box_loss', [0]*len(df)), label='val/box_loss', color='blue')
    axes[1].set_title('Box Loss: Train vs Validación')
    axes[1].set_xlabel('Época')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

---
## Sección 8: Validación del modelo entrenado

La validación calcula las métricas sobre el conjunto de **test** (datos que el modelo nunca vio durante el entrenamiento).

### Métricas clave para evaluar
- **mAP@50**: Mean Average Precision con IoU threshold=0.5. **Objetivo: > 0.70** para un buen detector de barcos
- **mAP@50-95**: Promedio sobre múltiples thresholds de IoU (más exigente). **Objetivo: > 0.45**
- **Precisión**: De todos los barcos detectados, ¿cuántos son realmente barcos? (minimiza falsos positivos)
- **Recall**: De todos los barcos existentes, ¿cuántos detecta el modelo? (minimiza falsos negativos)

> En aplicaciones de monitoreo marítimo, **el recall suele priorizarse sobre la precisión**: es peor no detectar un barco que existe (falso negativo) que detectar algo que no es un barco (falso positivo).

In [ ]:
# Validación sobre el conjunto de test
BEST_MODEL = f'{TRAIN_DIR}/weights/best.pt'

!yolo task=detect mode=val \
    model={BEST_MODEL} \
    data={dataset.location}/data.yaml \
    split=test \
    conf=0.25 \
    iou=0.5

print("\n✅ Validación completada")

---
## Sección 9: Inferencia con el modelo entrenado

Probamos el modelo sobre imágenes de test que **nunca vio durante el entrenamiento**.

In [ ]:
# Inferencia sobre imágenes de test
TEST_IMAGES_DIR = f'{dataset.location}/test/images'

!yolo task=detect mode=predict \
    model={BEST_MODEL} \
    conf=0.25 \
    iou=0.45 \
    source={TEST_IMAGES_DIR} \
    save=True \
    name=ship_detection_test

print("\n✅ Predicciones guardadas")

In [ ]:
import glob, os
from IPython.display import Image as IPyImage, display
import random

# Mostrar 6 predicciones aleatorias
predict_folder = max(
    glob.glob(f'{HOME}/runs/detect/ship_detection_test*/'),
    key=os.path.getmtime
)

result_images = glob.glob(f'{predict_folder}/*.jpg') + \
                glob.glob(f'{predict_folder}/*.png')

sample = random.sample(result_images, min(6, len(result_images)))

print(f"📸 Mostrando {len(sample)} predicciones de ejemplo:")
for i, img_path in enumerate(sample):
    print(f"\n--- Imagen {i+1}: {os.path.basename(img_path)} ---")
    display(IPyImage(filename=img_path, width=700))

In [ ]:
# Inferencia usando la API de Python (más control sobre los resultados)
from ultralytics import YOLO
import supervision as sv
from PIL import Image
import matplotlib.pyplot as plt
import glob, random

model = YOLO(BEST_MODEL)

# Elegir imagen aleatoria del test
test_images = glob.glob(f'{TEST_IMAGES_DIR}/*.jpg') + \
              glob.glob(f'{TEST_IMAGES_DIR}/*.png')

img_path = random.choice(test_images)
image = Image.open(img_path).convert('RGB')

# Predicción
result = model.predict(image, conf=0.25, iou=0.45)[0]

# Detecciones
detections = sv.Detections.from_ultralytics(result)

print(f"🛳️  Barcos detectados en {os.path.basename(img_path)}: {len(detections)}")
for i, (box, conf) in enumerate(zip(detections.xyxy, detections.confidence)):
    x1, y1, x2, y2 = box
    w, h = x2-x1, y2-y1
    print(f"  Barco {i+1}: bbox=({x1:.0f}, {y1:.0f}, {x2:.0f}, {y2:.0f}), "
          f"tamaño={w:.0f}×{h:.0f}px, confianza={conf:.2f}")

# Visualización con supervision
box_annotator   = sv.BoxAnnotator(thickness=2)
label_annotator = sv.LabelAnnotator(
    text_color=sv.Color.WHITE,
    text_scale=0.5
)
corner_annotator = sv.BoxCornerAnnotator(color=sv.Color.RED)

labels = [f"ship {c:.2f}" for c in detections.confidence]

annotated = image.copy()
annotated = box_annotator.annotate(annotated, detections=detections)
annotated = label_annotator.annotate(annotated, detections=detections, labels=labels)
annotated = corner_annotator.annotate(annotated, detections=detections)

sv.plot_image(annotated, size=(12, 12))

---
## Sección 10: Análisis de errores y ajuste de umbral de confianza

El umbral de confianza (`conf`) controla el tradeoff precisión/recall:
- **conf bajo** (0.1–0.2): detecta más barcos pero con más falsos positivos
- **conf alto** (0.5–0.7): más precisión pero puede perder barcos pequeños

Para aplicaciones de **monitoreo satelital** se recomienda un conf entre 0.25–0.40.

In [ ]:
from ultralytics import YOLO
import glob, random
import matplotlib.pyplot as plt
from PIL import Image

model = YOLO(BEST_MODEL)

# Evaluar cómo varía el número de detecciones con distintos thresholds
# (sobre un subconjunto de 20 imágenes)
test_sample = random.sample(
    glob.glob(f'{TEST_IMAGES_DIR}/*.jpg') + glob.glob(f'{TEST_IMAGES_DIR}/*.png'),
    min(20, len(test_images))
)

conf_values = [0.10, 0.20, 0.25, 0.30, 0.40, 0.50, 0.60]
avg_detections = []

for conf in conf_values:
    total = 0
    for img_path in test_sample:
        results = model.predict(img_path, conf=conf, verbose=False)
        total += len(results[0].boxes)
    avg_detections.append(total / len(test_sample))

plt.figure(figsize=(8, 5))
plt.plot(conf_values, avg_detections, marker='o', color='steelblue', linewidth=2)
plt.axvline(x=0.25, color='red', linestyle='--', alpha=0.7, label='conf=0.25 (recomendado)')
plt.xlabel('Umbral de confianza')
plt.ylabel('Detecciones promedio por imagen')
plt.title('Detecciones promedio vs. Umbral de confianza')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Sección 11: Despliegue del modelo en Roboflow

Una vez satisfechos con el modelo, lo desplegamos en Roboflow Deploy para tener un endpoint de inferencia en la nube, accesible desde cualquier dispositivo via API REST.

In [ ]:
# Subir modelo entrenado a Roboflow Deploy
print("🚀 Subiendo modelo a Roboflow Deploy...")

project.version(dataset.version).deploy(
    model_type="yolov11",
    model_path=TRAIN_DIR
)

print("✅ Modelo desplegado en Roboflow")
print(f"   Podés accederlo en: https://app.roboflow.com/{WORKSPACE_NAME}/{PROJECT_NAME}/{VERSION_NUMBER}")

In [ ]:
# Inferencia usando el modelo desplegado en Roboflow
!pip install inference --quiet

import inference
import supervision as sv
import cv2, random, os
import IPython

# ID del modelo desplegado
model_id = f"{PROJECT_NAME}/{VERSION_NUMBER}"
model_api = inference.get_model(model_id, ROBOFLOW_API_KEY)

# Correr inferencia sobre 4 imágenes de test
test_sample_api = random.sample(test_images, min(4, len(test_images)))

for img_path in test_sample_api:
    print(f"\n🛰️  Inferencia API: {os.path.basename(img_path)}")
    image = cv2.imread(img_path)

    results   = model_api.infer(image, confidence=0.25, overlap=30)[0]
    detections = sv.Detections.from_inference(results)

    print(f"   Barcos detectados: {len(detections)}")

    box_annotator   = sv.BoxAnnotator(thickness=2)
    label_annotator = sv.LabelAnnotator()

    annotated = box_annotator.annotate(scene=image.copy(), detections=detections)
    annotated = label_annotator.annotate(scene=annotated, detections=detections)

    _, buf = cv2.imencode('.jpg', annotated)
    IPython.display.display(IPython.display.Image(data=buf))

---
## Sección 12: Próximos pasos y mejoras posibles

### Para mejorar la performance del modelo

1. **Más datos anotados**: El factor más importante. Con 500+ imágenes bien anotadas el modelo mejora significativamente. Roboflow Universe tiene datasets de barcos satelitales que se pueden combinar.

2. **Mayor resolución de imagen**: Probar `imgsz=1280` si el modelo no detecta bien los barcos pequeños. Requiere `batch=8` para no saturar la VRAM.

3. **Modelo más grande**: Probar `yolo11l.pt` (large) si hay suficientes datos y el T4 lo soporta.

4. **Clases adicionales**: Distinguir tipos de embarcaciones: `cargo_ship`, `tanker`, `sailboat`, `fishing_boat`. Requiere más anotaciones específicas por clase.

5. **SAHI (Slicing Aided Hyper Inference)**: Para imágenes satelitales de alta resolución donde los barcos son muy pequeños, SAHI divide la imagen en tiles y los procesa individualmente:
   ```python
   pip install sahi
   from sahi import AutoDetectionModel
   from sahi.predict import get_sliced_prediction
   ```

6. **Pseudo-labeling**: Usar el modelo entrenado para pre-anotar nuevas imágenes automáticamente, revisar manualmente las predicciones y re-entrenar.

### Para productivizar
- Exportar a **ONNX** para inferencia eficiente sin dependencia de GPU: `model.export(format='onnx')`
- Exportar a **TensorRT** para inferencia optimizada en NVIDIA: `model.export(format='engine')`
- Armar un pipeline de monitoreo: descarga automática de nuevas imágenes satelitales → inferencia → alertas si se detectan barcos en zonas restringidas

---
**Recursos adicionales:**
- [Ultralytics YOLO11 Docs](https://docs.ultralytics.com)
- [Roboflow Universe — Ship Datasets](https://universe.roboflow.com/search?q=ship+satellite)
- [SAHI para objetos pequeños](https://github.com/obss/sahi)
- [Satellite Ship Detection Paper (SAR)](https://arxiv.org/abs/1901.01124)